In [ ]:
# Python ≥3.8 is required
import sys
assert sys.version_info >= (3, 8)

# Scikit-Learn ≥ 1.0 is required
import sklearn
assert sklearn.__version__ >= "1.0"

# Common imports
import numpy as np
import pandas as pd
import os

# To plot figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

# Precision options
np.set_printoptions(precision=2)
pd.options.display.float_format = '{:.2f}'.format

In [ ]:
# Python ≥3.9 is required
import sys
assert sys.version_info >= (3, 9)

import seaborn as sns

# Statistical analysis and testing
from scipy import stats
from statsmodels.formula.api import ols
import statsmodels.api as sm

# Ignore useless warnings (see SciPy issue #5998)
import warnings
warnings.filterwarnings(action="ignore", message="^internal gelsd")

### Regression Prepare

In [ ]:
train_import = pd.read_csv("train_bin_nhanes.csv")
test_import = pd.read_csv("test_bin_nhanes.csv")

In [ ]:
X_train = train_import.drop(columns=["target"])
y_train = train_import["target"]

X_test = test_import.drop(columns=["target"])
y_test = test_import["target"]

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(y_train.unique())

In [ ]:
# add noise variables for extreme overfitting
for i in range(934):
    X_train[f"noise{i}"] = np.random.normal(size=len(X_train))
    X_test[f"noise{i}"] = np.random.normal(size=len(X_test))

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(y_train.unique())

## Model 2 - Regression

In [ ]:
from sklearn.model_selection import cross_validate, KFold
n_splits = 10
k_fold = KFold(n_splits=n_splits, shuffle=True, random_state=42)

In [ ]:
def display_scores(scores):
    print("Scores:", scores)
    print("Mean:", f"{scores.mean():.2f}")
    print("Standard deviation:", f"{scores.std():.2f}")

#### Logistic Regression

In [ ]:
#for overfitting
from sklearn.linear_model import LogisticRegression

plain_logreg = LogisticRegression(
    penalty=None,
    max_iter=200,
    random_state=77
)

plain_logreg.fit(X_train, y_train)

In [ ]:
#penalty terms to create moderate overfitting
from sklearn.linear_model import LogisticRegression

plain_logreg = LogisticRegression(
    penalty="l2",
    C=0.1,
    max_iter=200,
    random_state=77
)

plain_logreg.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import roc_auc_score

y_probs_plain_logreg = plain_logreg.predict_proba(X_test)[:, 1]
test_auc_plain_logreg = roc_auc_score(y_test, y_probs_plain_logreg)

print("Test AUC:", test_auc_plain_logreg)

In [ ]:
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

fpr, tpr, _ = roc_curve(y_test, y_probs_plain_logreg)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"AUC = {test_auc_plain_logreg:.3f}")
plt.plot([0,1],[0,1],"--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Plain Logistic Regression")
plt.legend()
plt.show()

In [ ]:
train_probs_plain_logreg= plain_logreg.predict_proba(X_train)[:, 1]
train_auc_plain_logreg = roc_auc_score(y_train, train_probs_plain_logreg)

print("Train AUC:", train_auc_plain_logreg)

In [ ]:
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

fpr, tpr, _ = roc_curve(y_train, train_probs_plain_logreg)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"AUC = {train_auc_plain_logreg:.3f}")
plt.plot([0,1],[0,1],"--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Plain Logistic Regression")
plt.legend()
plt.show()

In [ ]:
import numpy as np
from sklearn.utils import resample
from sklearn.metrics import roc_auc_score

In [ ]:
def bootstrap_optimism_auc(model, X_train, y_train, B=50):

    apparent_scores = []
    test_scores = []
    optimisms = []

    X_train = np.array(X_train)
    y_train = np.array(y_train)

    for i in range(B):

        # bootstrap sample
        X_boot, y_boot = resample(X_train, y_train, replace=True)

        model.fit(X_boot, y_boot)

        # predictions
        boot_probs = model.predict_proba(X_boot)[:, 1]
        full_probs = model.predict_proba(X_train)[:, 1]

        boot_auc = roc_auc_score(y_boot, boot_probs)
        test_auc = roc_auc_score(y_train, full_probs)

        optimism = boot_auc - test_auc

        apparent_scores.append(boot_auc)
        test_scores.append(test_auc)
        optimisms.append(optimism)

    avg_optimism = np.mean(optimisms)

    # final fit on full training data
    model.fit(X_train, y_train)

    apparent_final = roc_auc_score(
        y_train,
        model.predict_proba(X_train)[:, 1]
    )

    corrected = apparent_final - avg_optimism

    return {
        "apparent": apparent_final,
        "optimism": avg_optimism,
        "corrected": corrected
    }

In [ ]:
plain_logreg_results = bootstrap_optimism_auc(
    plain_logreg,
    X_train,
    y_train,
    B=50  # Number of bootstrap iterations
)

print("Plain Logisitc Bootstrap Optimism Results:", plain_logreg_results)

In [ ]:
print("Plain Bootstrap Optimism Results:", plain_logreg_results)